# Day 15 — Ensemble methods: Random Forest

Run each cell with **Shift + Enter**, from the top. We use our 150-student dataset, `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Introduction to Random Forest

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-015/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import numpy as np
import pandas as pd

# Bootstrap sampling: draw 10 students WITH replacement from 10. Some appear twice, some not at all
rng = np.random.default_rng(1)
students = [f"S{i}" for i in range(1, 11)]
sample = [str(s) for s in rng.choice(students, size=10, replace=True)]
print("Bootstrap sample:", sample)
counts = pd.Series(sample).value_counts()
print("Drawn more than once:", list(counts[counts > 1].index))
print("Left out (out-of-bag):", [s for s in students if s not in set(sample)])
# On average a bootstrap sample leaves out about 1/e = 37% of the records
left = [1 - len(set(rng.choice(127, 127))) / 127 for _ in range(1000)]
print(f"Share of 127 students left out, averaged over 1,000 samples: {np.mean(left):.1%}")

Bootstrap sample: ['S5', 'S6', 'S8', 'S10', 'S1', 'S2', 'S9', 'S10', 'S3', 'S4']
Drawn more than once: ['S10']
Left out (out-of-bag): ['S7']
Share of 127 students left out, averaged over 1,000 samples: 36.7%


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# Your first Random Forest on our students: the same 127 / 23 split as Day 14 (the test set waits for Section 5)
df = pd.read_csv("student_data_150.csv")
features = ["StudyHours", "Attendance", "PreviousScore"]
train_val, test = train_test_split(df, test_size=0.15, random_state=42, stratify=df["Passed"])
y_tv = train_val["Passed"].astype(int)
forest = RandomForestClassifier(n_estimators=100, random_state=42).fit(train_val[features], y_tv)

# Five of its trees vote on a NEW student: 2.5 hours of study, 76% attendance, previous score 55
student = pd.DataFrame([[2.5, 76, 55]], columns=features)
votes = ["Pass" if t.predict(student.values)[0] == 1 else "Fail" for t in forest.estimators_[:5]]
for i, v in enumerate(votes, 1):
    print(f"Tree {i}: {v}")
print(f"Five trees: {votes.count('Pass')} pass, {votes.count('Fail')} fail -> majority: {max(set(votes), key=votes.count)}")
all_votes = np.array([t.predict(student.values)[0] for t in forest.estimators_])
print(f"All 100 trees: {int(all_votes.sum())} pass, {100 - int(all_votes.sum())} fail -> the forest predicts "
      f"{'Pass' if forest.predict(student)[0] == 1 else 'Fail'}")

Tree 1: Fail
Tree 2: Pass
Tree 3: Pass
Tree 4: Pass
Tree 5: Pass
Five trees: 4 pass, 1 fail -> majority: Pass
All 100 trees: 74 pass, 26 fail -> the forest predicts Pass


In [ ]:
# One tree vs a forest, compared fairly: 5-fold cross-validation on the same 127 students
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, m in [("Decision tree (unlimited)", DecisionTreeClassifier(random_state=42)),
                ("Decision tree (max_depth=2)", DecisionTreeClassifier(max_depth=2, random_state=42)),
                ("Random forest (100 trees)", RandomForestClassifier(n_estimators=100, random_state=42))]:
    s = cross_val_score(m, train_val[features], y_tv, cv=cv5)
    print(f"{name:<28} CV accuracy {s.mean():.3f} +/- {s.std():.3f} | folds {np.round(s, 2).tolist()}")
print(f"Always guessing 'pass': {y_tv.mean():.3f}")

Decision tree (unlimited)    CV accuracy 0.717 +/- 0.065 | folds [0.69, 0.65, 0.72, 0.68, 0.84]
Decision tree (max_depth=2)  CV accuracy 0.740 +/- 0.039 | folds [0.73, 0.73, 0.76, 0.68, 0.8]


Random forest (100 trees)    CV accuracy 0.804 +/- 0.063 | folds [0.69, 0.81, 0.8, 0.84, 0.88]
Always guessing 'pass': 0.709


In [ ]:
# Stability: retrain on 20 different bootstrap samples of the 127 and compare the predictions for the 23 held-out
# students (their features only; their results stay unseen until Section 5). A single tree flips its mind far more than a forest does
rng = np.random.default_rng(0)
tree_preds, forest_preds = [], []
for seed in range(20):
    idx = rng.choice(len(train_val), len(train_val))
    Xb, yb = train_val[features].iloc[idx], y_tv.iloc[idx]
    tree_preds.append(DecisionTreeClassifier(random_state=seed).fit(Xb, yb).predict(test[features]))
    forest_preds.append(RandomForestClassifier(n_estimators=100, random_state=seed).fit(Xb, yb).predict(test[features]))
def unstable(preds):
    p = np.array(preds).mean(axis=0)                 # share of the 20 models predicting pass, per student
    return int(((p > 0.1) & (p < 0.9)).sum())        # students the models disagree on (10-90% pass)
print(f"Test students on whom the 20 models disagree: single tree {unstable(tree_preds)} of 23 | forest {unstable(forest_preds)} of 23")

Test students on whom the 20 models disagree: single tree 8 of 23 | forest 4 of 23


In [ ]:
# Practice: 50, 100 and 200 trees (cross-validated accuracy on the 127)
for n in [50, 100, 200]:
    s = cross_val_score(RandomForestClassifier(n_estimators=n, random_state=42), train_val[features], y_tv, cv=cv5)
    print(f"n_estimators={n:<4} CV accuracy {s.mean():.3f} +/- {s.std():.3f}")

n_estimators=50   CV accuracy 0.827 +/- 0.051


n_estimators=100  CV accuracy 0.804 +/- 0.063


n_estimators=200  CV accuracy 0.804 +/- 0.071


## 2. How random forest works: bagging and feature sampling

In [ ]:
# Section 2: bagging, step by step. Two bootstrap samples of six students (with replacement)
six = pd.DataFrame({"Student": [1, 2, 3, 4, 5, 6], "Score": [65, 72, 80, 55, 90, 76]})
rng = np.random.default_rng(7)
for k in (1, 2):
    pick = six.sample(n=6, replace=True, random_state=int(rng.integers(1000)))
    missing = sorted(set(six["Student"]) - set(pick["Student"]))
    print(f"Bootstrap sample {k}: students {sorted(pick['Student'].tolist())} | not drawn: {missing}")

Bootstrap sample 1: students [1, 2, 2, 3, 3, 6] | not drawn: [4, 5]
Bootstrap sample 2: students [1, 1, 2, 4, 5, 5] | not drawn: [3, 6]


In [ ]:
from collections import Counter

# Random feature sampling: which question does each tree ask FIRST? We have 3 features, so max_features="sqrt"
# (and "log2") lets a split consider only 1 of the 3; max_features=None considers all 3
for mf in [None, "sqrt"]:
    rf = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=42).fit(train_val[features], y_tv)
    roots = Counter(features[t.tree_.feature[0]] for t in rf.estimators_)
    preds = np.array([t.predict(test[features].values) for t in rf.estimators_])     # 100 trees x 23 held-out students
    agree = np.mean([(preds[i] == preds[j]).mean() for i in range(100) for j in range(i + 1, 100)])
    print(f"max_features={str(mf):<5} root questions: {dict(roots.most_common())} | two trees agree on {agree:.0%} of students")

max_features=None  root questions: {'StudyHours': 63, 'PreviousScore': 27, 'Attendance': 10} | two trees agree on 79% of students


max_features=sqrt  root questions: {'StudyHours': 36, 'Attendance': 36, 'PreviousScore': 28} | two trees agree on 77% of students


In [ ]:
# The experiment: number of trees x max_features, by 5-fold cross-validation on the 127 (not the test set)
rows = []
for n in [10, 50, 100, 200]:
    for mf in ["sqrt", None]:
        s = cross_val_score(RandomForestClassifier(n_estimators=n, max_features=mf, random_state=42, n_jobs=-1),
                            train_val[features], y_tv, cv=cv5)
        rows.append({"Trees": n, "MaxFeatures": str(mf), "CVAccuracy": round(s.mean(), 3), "CVStd": round(s.std(), 3)})
experiment = pd.DataFrame(rows)
print(experiment.to_string(index=False))
print('("log2" gives the same forest as "sqrt" here: both mean 1 of 3 features)')
experiment.to_csv("forest_experiment.csv", index=False)

 Trees MaxFeatures  CVAccuracy  CVStd
    10        sqrt       0.772  0.050
    10        None       0.717  0.068
    50        sqrt       0.827  0.051
    50        None       0.749  0.082
   100        sqrt       0.804  0.063
   100        None       0.789  0.085
   200        sqrt       0.804  0.071
   200        None       0.788  0.066
("log2" gives the same forest as "sqrt" here: both mean 1 of 3 features)


In [ ]:
# A free validation score: each tree is checked on the students its bootstrap sample left out (out-of-bag)
for n in [10, 50, 100, 200]:
    rf = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1).fit(train_val[features], y_tv)
    print(f"n_estimators={n:<4} out-of-bag accuracy {rf.oob_score_:.3f}")

n_estimators=10   out-of-bag accuracy 0.717
n_estimators=50   out-of-bag accuracy 0.740


n_estimators=100  out-of-bag accuracy 0.748


n_estimators=200  out-of-bag accuracy 0.748


## 3. Key hyperparameters

In [ ]:
import time
from sklearn.model_selection import cross_validate

# Section 3: one parameter at a time, the others at their defaults (100 trees, "sqrt"). 5-fold CV on the 127
def check(label, **params):
    cv = cross_validate(RandomForestClassifier(random_state=42, n_jobs=-1, **params), train_val[features], y_tv,
                        cv=cv5, scoring=["accuracy", "f1"], return_train_score=True)
    print(f"{label:<22} train {cv['train_accuracy'].mean():.3f} | CV accuracy {cv['test_accuracy'].mean():.3f} "
          f"| CV F1 {cv['test_f1'].mean():.3f} | fit {cv['fit_time'].mean():.2f}s")
    return cv["test_accuracy"].mean()

depth_rows = [{"MaxDepth": str(d), "CVAccuracy": round(check(f"max_depth={d}", max_depth=d), 3)} for d in [3, 5, 10, None]]
pd.DataFrame(depth_rows).to_csv("forest_depth.csv", index=False)

max_depth=3            train 0.856 | CV accuracy 0.796 | CV F1 0.864 | fit 0.14s


max_depth=5            train 0.935 | CV accuracy 0.812 | CV F1 0.873 | fit 0.13s


max_depth=10           train 1.000 | CV accuracy 0.804 | CV F1 0.867 | fit 0.13s


max_depth=None         train 1.000 | CV accuracy 0.804 | CV F1 0.867 | fit 0.14s


In [ ]:
# min_samples_leaf: bigger leaves, simpler trees
for leaf in [1, 2, 5, 10]:
    check(f"min_samples_leaf={leaf}", min_samples_leaf=leaf)
# bootstrap: False trains every tree on all 127 students, so only feature sampling makes them differ
for b in [True, False]:
    check(f"bootstrap={b}", bootstrap=b)

min_samples_leaf=1     train 1.000 | CV accuracy 0.804 | CV F1 0.867 | fit 0.12s


min_samples_leaf=2     train 0.937 | CV accuracy 0.812 | CV F1 0.875 | fit 0.12s


min_samples_leaf=5     train 0.841 | CV accuracy 0.788 | CV F1 0.861 | fit 0.13s


min_samples_leaf=10    train 0.799 | CV accuracy 0.764 | CV F1 0.845 | fit 0.13s


bootstrap=True         train 1.000 | CV accuracy 0.804 | CV F1 0.867 | fit 0.14s


bootstrap=False        train 1.000 | CV accuracy 0.764 | CV F1 0.836 | fit 0.10s


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score

# The practical starting configuration, then the lesson's grid search (5-fold CV, scored by F1 for pass)
check("starting configuration", n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=2, max_features="sqrt")
param_grid = {"n_estimators": [50, 100, 200], "max_depth": [5, 10, None], "min_samples_split": [2, 5, 10],
              "min_samples_leaf": [1, 2, 5], "max_features": ["sqrt", "log2"]}
start = time.time()
grid = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid, cv=cv5, scoring="f1", n_jobs=-1)
grid.fit(train_val[features], y_tv)
print(f"Grid: {len(grid.cv_results_['params'])} combinations x 5 folds in {time.time() - start:.0f}s")
print("Best parameters:", grid.best_params_)
print(f"Best CV F1: {grid.best_score_:.3f}")
top = np.sort(grid.cv_results_["mean_test_score"])[::-1]
print(f"Combinations within 0.01 of the best: {(top >= grid.best_score_ - 0.01).sum()}")
fail_recall = cross_val_score(grid.best_estimator_, train_val[features], y_tv, cv=cv5, scoring=make_scorer(recall_score, pos_label=0))
print(f"Best model's CV recall for fail: {fail_recall.mean():.3f}")

starting configuration train 0.917 | CV accuracy 0.812 | CV F1 0.875 | fit 0.13s


Grid: 162 combinations x 5 folds in 38s
Best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 50}
Best CV F1: 0.884
Combinations within 0.01 of the best: 40


Best model's CV recall for fail: 0.568


## 4. Random forest implementation in Python

In [ ]:
# Section 4: implementation, end to end. 1. Load and explore
df = pd.read_csv("student_data_150.csv")
df["Result"] = np.where(df["Passed"], "Pass", "Fail")
features = ["StudyHours", "Attendance", "PreviousScore"]
print(f"Shape {df.shape} | missing values in features {df[features].isna().sum().sum()}")
print(df[features + ["Result"]].head(3).to_string(index=False))
print(df["Result"].value_counts().to_string())
print(f"Pass {df['Passed'].mean():.1%}: always guessing 'Pass' would already score {df['Passed'].mean():.3f} accuracy")

Shape (150, 10) | missing values in features 0
 StudyHours  Attendance  PreviousScore Result
        1.0        60.0             40   Fail
        2.0        70.0             50   Fail
        3.0        80.0             60   Pass
Result
Pass    106
Fail     44
Pass 70.7%: always guessing 'Pass' would already score 0.707 accuracy


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

# 2-3. Same split as before (23 test students kept for Section 5). Text labels work directly in scikit-learn.
X_tv, y_lab = train_val[features], train_val["Passed"].map({True: "Pass", False: "Fail"})
rf = RandomForestClassifier(n_estimators=100, max_features="sqrt", random_state=42, n_jobs=-1)
# Baseline, judged on out-of-fold predictions: each of the 127 is predicted by a forest that never saw it
oof_rf = cross_val_predict(rf, X_tv, y_lab, cv=cv5)
print(classification_report(y_lab, oof_rf, digits=3).rstrip())

              precision    recall  f1-score   support

        Fail      0.714     0.541     0.615        37
        Pass      0.828     0.911     0.868        90

    accuracy                          0.803       127
   macro avg      0.771     0.726     0.742       127
weighted avg      0.795     0.803     0.794       127


In [ ]:
from sklearn.metrics import f1_score

# 4. Tune with GridSearchCV on the 127 (the lesson's grid, scored by F1 for "Pass")
param_grid = {"n_estimators": [50, 100, 200], "max_depth": [5, 10, None], "min_samples_split": [2, 5],
              "min_samples_leaf": [1, 2, 5], "max_features": ["sqrt", "log2"]}
grid = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid, cv=cv5,
                    scoring=make_scorer(f1_score, pos_label="Pass"), n_jobs=-1).fit(X_tv, y_lab)
best_model = grid.best_estimator_
print(f"{len(grid.cv_results_['params'])} combinations | best parameters: {grid.best_params_}")
print(f"Best CV F1 (pass): {grid.best_score_:.3f}")

108 combinations | best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 50}
Best CV F1 (pass): 0.884


In [ ]:
from sklearn.metrics import accuracy_score, precision_score

# 5. Compare the three models on the SAME records: out-of-fold predictions for all 127 students
models = {"Decision tree (max_depth=5)": DecisionTreeClassifier(max_depth=5, random_state=42),
          "Random forest (baseline)": rf,
          "Random forest (tuned)": RandomForestClassifier(random_state=42, n_jobs=-1, **grid.best_params_)}
rows = []
for name, m in models.items():
    p = cross_val_predict(m, X_tv, y_lab, cv=cv5)
    rows.append({"Model": name, "Accuracy": accuracy_score(y_lab, p), "Precision (pass)": precision_score(y_lab, p, pos_label="Pass"),
                 "Recall (fail)": recall_score(y_lab, p, pos_label="Fail"), "F1 (pass)": f1_score(y_lab, p, pos_label="Pass")})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("(The tuned settings were chosen on these same 127 students, so its row is slightly optimistic.)")

                      Model  Accuracy  Precision (pass)  Recall (fail)  F1 (pass)
Decision tree (max_depth=5)     0.709             0.791          0.486      0.796
   Random forest (baseline)     0.803             0.828          0.541      0.868
      Random forest (tuned)     0.827             0.840          0.568      0.884
(The tuned settings were chosen on these same 127 students, so its row is slightly optimistic.)


In [ ]:
from sklearn.inspection import permutation_importance

# 6. Feature importance: impurity-based (from training) and permutation (on held-out folds)
importance = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=False)
print("Impurity-based:", importance.round(3).to_dict())
perm = {f: [] for f in features}
for tr, va in cv5.split(X_tv, y_lab):
    m = RandomForestClassifier(random_state=42, n_jobs=-1, **grid.best_params_).fit(X_tv.iloc[tr], y_lab.iloc[tr])
    r = permutation_importance(m, X_tv.iloc[va], y_lab.iloc[va], n_repeats=20, random_state=42)
    for f, v in zip(features, r.importances_mean):
        perm[f].append(v)
perm_mean = pd.Series({f: np.mean(v) for f, v in perm.items()}).sort_values(ascending=False)
print("Permutation (accuracy drop when shuffled, held-out):", perm_mean.round(3).to_dict())
pd.DataFrame({"Feature": importance.index, "Impurity": importance.values.round(3),
              "Permutation": perm_mean.reindex(importance.index).values.round(3)}).to_csv("forest_importance.csv", index=False)

Impurity-based: {'PreviousScore': 0.387, 'StudyHours': 0.33, 'Attendance': 0.283}


Permutation (accuracy drop when shuffled, held-out): {'StudyHours': 0.111, 'PreviousScore': 0.087, 'Attendance': 0.065}


## 5. Evaluation and comparison

In [ ]:
from sklearn.metrics import confusion_matrix

# Section 5: the ONE test. Fit the three models on the 127, then predict the 23 students kept aside. Fail is the
# positive class: the school wants to find failing students
X_te, y_te = test[features], test["Passed"].map({True: "Pass", False: "Fail"})
final = {"Decision tree (max_depth=5)": DecisionTreeClassifier(max_depth=5, random_state=42),
         "Random forest (baseline)": RandomForestClassifier(n_estimators=100, max_features="sqrt", random_state=42, n_jobs=-1),
         "Random forest (tuned)": RandomForestClassifier(random_state=42, n_jobs=-1, **grid.best_params_)}
print(f"{len(y_te)} test students: {(y_te == 'Fail').sum()} fail, {(y_te == 'Pass').sum()} pass")
for name, m in final.items():
    pred = m.fit(X_tv, y_lab).predict(X_te)
    print(f"\n{name}: accuracy {accuracy_score(y_te, pred):.3f} | precision (fail) {precision_score(y_te, pred, pos_label='Fail', zero_division=0):.3f}"
          f" | recall (fail) {recall_score(y_te, pred, pos_label='Fail'):.3f} | F1 (fail) {f1_score(y_te, pred, pos_label='Fail'):.3f}")
    cm = confusion_matrix(y_te, pred, labels=["Fail", "Pass"])
    print(f"  actual Fail: {cm[0, 0]} predicted Fail, {cm[0, 1]} predicted Pass | actual Pass: {cm[1, 0]} predicted Fail, {cm[1, 1]} predicted Pass")

23 test students: 7 fail, 16 pass

Decision tree (max_depth=5): accuracy 0.870 | precision (fail) 0.750 | recall (fail) 0.857 | F1 (fail) 0.800
  actual Fail: 6 predicted Fail, 1 predicted Pass | actual Pass: 2 predicted Fail, 14 predicted Pass

Random forest (baseline): accuracy 0.739 | precision (fail) 0.600 | recall (fail) 0.429 | F1 (fail) 0.500
  actual Fail: 3 predicted Fail, 4 predicted Pass | actual Pass: 2 predicted Fail, 14 predicted Pass



Random forest (tuned): accuracy 0.783 | precision (fail) 0.750 | recall (fail) 0.429 | F1 (fail) 0.545
  actual Fail: 3 predicted Fail, 4 predicted Pass | actual Pass: 1 predicted Fail, 15 predicted Pass


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# ROC and AUC with Fail as the positive class: how well each model RANKS failing students, at every threshold
y_fail = (y_te == "Fail").astype(int)
for name, m in final.items():
    p_fail = m.predict_proba(X_te)[:, list(m.classes_).index("Fail")]
    fpr, tpr, _ = roc_curve(y_fail, p_fail)
    print(f"{name:<28} AUC {roc_auc_score(y_fail, p_fail):.3f} | distinct probabilities {len(np.unique(p_fail))}")
    key = name.split(" (")[0].replace(" ", "_").lower() + ("_tuned" if "tuned" in name else "")
    pd.DataFrame({"FalsePositiveRate": fpr.round(3), "TruePositiveRate": tpr.round(3)}).to_csv(f"roc_{key}.csv", index=False)

Decision tree (max_depth=5)  AUC 0.888 | distinct probabilities 5
Random forest (baseline)     AUC 0.879 | distinct probabilities 19
Random forest (tuned)        AUC 0.902 | distinct probabilities 23
